# Python Web Scraping with Selenium — Screener.in
### Hands-on Workshop Notebook

---

**Contents**

| # | Module | Topic |
|---|--------|-------|
| 1 | WebDriver Basics | Launch, navigate, scroll, Chrome options |
| 2 | Class-Based Session | Search Bajaj Finance, explore company page, open peer tab |
| 3 | Customizing Sessions | Same flow for HDFC Bank with exception handling |
| 4 | Locating Elements | ID · Name · Class · CSS · XPath · Link Text · Partial Link Text |
| 5 | Financial Data Scraper | Multi-company scraper → `financial_data.csv` |

---
> **Before you start:** make sure `selenium` and `beautifulsoup4` are installed.
> ```
> pip install selenium beautifulsoup4
> ```
> ChromeDriver must match your installed Chrome version and be on your PATH.

---
## Module 1 — WebDriver Basics

Launch a Chrome browser, navigate to **Screener.in**, and practise scrolling.
At the end of the module we look at `ChromeOptions` for incognito and headless modes.

In [ ]:
from selenium import webdriver

In [ ]:
# Launch Chrome and navigate to Screener.in
driver = webdriver.Chrome()
driver.get("https://www.screener.in")

### Scrolling

In [ ]:
# Scroll down by 500 pixels
driver.execute_script("window.scrollBy(0, 500)")

In [ ]:
# Scroll to the very bottom of the page
driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")

In [ ]:
# Scroll back to the top
driver.execute_script("window.scrollTo(0, 0)")

In [ ]:
driver.quit()

### Chrome Options

In [ ]:
options = webdriver.ChromeOptions()
options.add_argument("--incognito")          # private session
options.add_argument("--window-size=1920,1080")
options.add_argument("--headless")           # no visible browser window

driver = webdriver.Chrome(options=options)
driver.get("https://www.screener.in")
driver.quit()

---
## Module 2 — Class-Based Session (Bajaj Finance)

A **test class** with `setup_method` / `teardown_method` lifecycle hooks keeps the
WebDriver properly initialised and torn down even when a test fails.

**Flow:** Screener.in homepage → search "Bajaj Finance" → autocomplete → company page
→ open a peer company in a new tab → switch tabs → close both.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


class TestScreener():

    def setup_method(self, method):
        self.driver = webdriver.Chrome()
        self.vars = {}

    def teardown_method(self, method):
        self.driver.quit()

    # Utility: wait for a new browser tab/window to appear
    def wait_for_window(self, timeout=2):
        time.sleep(round(timeout / 1000))
        wh_now  = self.driver.window_handles
        wh_then = self.vars["window_handles"]
        if len(wh_now) > len(wh_then):
            return set(wh_now).difference(set(wh_then)).pop()

    def test_search_bajaj_finance(self):
        self.driver.get("https://www.screener.in")
        self.driver.set_window_size(1920, 1080)

        # Scroll the homepage slightly, then return to top
        self.driver.execute_script("window.scrollTo(0, 300)")
        self.driver.execute_script("window.scrollTo(0, 0)")

        # Type "Bajaj Finance" in the search box
        self.driver.find_element(By.NAME, "q").click()
        self.driver.find_element(By.NAME, "q").send_keys("Bajaj Finance")

        # Click the first autocomplete suggestion
        first_suggestion = WebDriverWait(self.driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "ul.awesomplete > li:first-child"))
        )
        first_suggestion.click()

        # Wait until the P&L section is present — confirms page has loaded
        WebDriverWait(self.driver, 10).until(
            EC.presence_of_element_located((By.ID, "profit-loss"))
        )

        # Scroll through the company page
        self.driver.execute_script("window.scrollTo(0, 500)")
        self.driver.execute_script("window.scrollTo(0, 1500)")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        self.driver.execute_script("window.scrollTo(0, 0)")

        # Locate the first peer company link in the Peer Comparison section
        peer_link = WebDriverWait(self.driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "#peers table tbody tr:first-child a")
            )
        )
        self.driver.execute_script("arguments[0].scrollIntoView();", peer_link)

        # Save handles, then open the peer company page in a new tab
        self.vars["window_handles"] = self.driver.window_handles
        peer_url = peer_link.get_attribute("href")
        self.driver.execute_script("window.open(arguments[0], '_blank');", peer_url)

        # Switch to the new tab
        self.vars["peer_tab"] = self.wait_for_window(2000)
        self.vars["root"]     = self.driver.current_window_handle
        self.driver.switch_to.window(self.vars["peer_tab"])

        self.driver.execute_script("window.scrollTo(0, 1000)")

        # Close peer tab and return to Bajaj Finance page
        self.driver.close()
        self.driver.switch_to.window(self.vars["root"])
        self.driver.close()

In [ ]:
testClass = TestScreener()
testClass.setup_method("")
testClass.test_search_bajaj_finance()
testClass.teardown_method("")

---
## Module 3 — Customizing Sessions (HDFC Bank)

The same class structure, now targeting **HDFC Bank**.
A `try / except NoSuchElementException` block gracefully handles pages where
the Peer Comparison table may not be present.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException


class TestScreener():

    def setup_method(self, method):
        self.driver = webdriver.Chrome()
        self.vars = {}

    def teardown_method(self, method):
        self.driver.quit()

    def wait_for_window(self, timeout=2):
        time.sleep(round(timeout / 1000))
        wh_now  = self.driver.window_handles
        wh_then = self.vars["window_handles"]
        if len(wh_now) > len(wh_then):
            return set(wh_now).difference(set(wh_then)).pop()

    def test_search_hdfc_bank(self):
        self.driver.get("https://www.screener.in")
        self.driver.set_window_size(1920, 1080)

        self.driver.execute_script("window.scrollTo(0, 300)")
        self.driver.execute_script("window.scrollTo(0, 0)")

        # Search for HDFC Bank
        self.driver.find_element(By.NAME, "q").click()
        self.driver.find_element(By.NAME, "q").send_keys("HDFC Bank")

        first_suggestion = WebDriverWait(self.driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "ul.awesomplete > li:first-child"))
        )
        first_suggestion.click()

        WebDriverWait(self.driver, 10).until(
            EC.presence_of_element_located((By.ID, "profit-loss"))
        )

        self.driver.execute_script("window.scrollTo(0, 500)")
        self.driver.execute_script("window.scrollTo(0, 1500)")
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        self.driver.execute_script("window.scrollTo(0, 0)")

        self.vars["window_handles"] = self.driver.window_handles

        try:
            peer_link = self.driver.find_element(
                By.CSS_SELECTOR, "#peers table tbody tr:first-child a"
            )
            self.driver.execute_script("arguments[0].scrollIntoView();", peer_link)
            peer_url = peer_link.get_attribute("href")
            self.driver.execute_script("window.open(arguments[0], '_blank');", peer_url)

            self.vars["peer_tab"] = self.wait_for_window(2000)
            self.vars["root"]     = self.driver.current_window_handle
            self.driver.switch_to.window(self.vars["peer_tab"])
            self.driver.execute_script("window.scrollTo(0, 1000)")
            self.driver.close()
            self.driver.switch_to.window(self.vars["root"])

        except NoSuchElementException:
            # Fallback when peers table is absent — jump to Quarterly Results
            quarterly_link = self.driver.find_element(By.LINK_TEXT, "Quarterly Results")
            self.driver.execute_script("arguments[0].scrollIntoView();", quarterly_link)
            quarterly_link.click()

        self.driver.close()

In [ ]:
testClass = TestScreener()
testClass.setup_method("")
testClass.test_search_hdfc_bank()
testClass.teardown_method("")

---
## Module 4 — Locating Elements on a Page

Selenium provides seven `By` strategies.
All examples below target elements on the **Bajaj Finance company page**.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

chromeOptions = Options()
chromeOptions.add_argument("--kiosk")

### 1 · By.ID

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in/company/BAJFINANCE/")

# Each major section on the Screener company page has a unique HTML id
pl_section = driver.find_element(By.ID, "profit-loss")
driver.execute_script("arguments[0].scrollIntoView();", pl_section)
time.sleep(1)

driver.quit()

### 2 · By.NAME

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in")

# ⚠ find_element (singular) returns the FIRST match — which may be a hidden element.
# Interacting with a hidden element raises ElementNotInteractableException.
input_element = driver.find_element(By.NAME, "q")
input_element.clear()          # may raise ElementNotInteractableException
input_element.send_keys("Reliance Industries")
input_element.send_keys(Keys.RETURN)

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in")

# Fix: use find_elements (plural) and filter for visible + enabled
search_elements = driver.find_elements(By.NAME, "q")

for element in search_elements:
    if element.is_displayed() and element.is_enabled():
        element.click()

element.clear()
element.send_keys("Reliance Industries")
element.send_keys(Keys.RETURN)

### 3 · By.CLASS_NAME

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in/company/BAJFINANCE/")

WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "profit-loss"))
)

# By.CLASS_NAME accepts exactly ONE class name (no spaces).
# For multi-class elements use By.CSS_SELECTOR instead (see next section).
ratio_items = driver.find_elements(By.CLASS_NAME, "flex")

count = 0
for item in ratio_items:
    if item.is_displayed() and item.text.strip():
        print(item.text)
        count += 1
    if count >= 5:
        break

driver.quit()

### 4 · By.CSS_SELECTOR

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in/company/BAJFINANCE/")

WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "profit-loss"))
)

# CSS Selector supports: tag, .class, #id, [attr], pseudo-classes, hierarchy
# More expressive than CLASS_NAME — use it whenever you need precision.

# Example 1: first ratio item inside the #top-ratios list
first_ratio = driver.find_element(By.CSS_SELECTOR, "#top-ratios li:first-child")
print("First ratio:", first_ratio.text)

# Example 2: search input located by attribute value
search_input = driver.find_element(By.CSS_SELECTOR, "input[name='q']")
print("Search box visible:", search_input.is_displayed())

driver.quit()

### 5 · By.XPATH

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in/company/BAJFINANCE/")

WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "profit-loss"))
)

# XPath can locate elements by tag, attribute, text content, or position.
# contains() is especially useful for financial tables where row labels vary.

# Locate the search box
driver.find_element(By.XPATH, '//input[@name="q"]').click()
time.sleep(1)

# Locate the "Net Profit" row inside the P&L table
net_profit_cell = driver.find_element(
    By.XPATH, "//section[@id='profit-loss']//td[contains(text(), 'Net Profit')]"
)
driver.execute_script("arguments[0].scrollIntoView();", net_profit_cell)
print("Row label:", net_profit_cell.text)

driver.quit()

### 6 · By.LINK_TEXT

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in/company/BAJFINANCE/")

wait = WebDriverWait(driver, 10)

# Matches the <a> tag whose complete visible text equals the string
quarterly_tab = wait.until(
    EC.element_to_be_clickable((By.LINK_TEXT, "Quarterly Results"))
)
driver.execute_script("arguments[0].scrollIntoView();", quarterly_tab)
quarterly_tab.click()

### 7 · By.PARTIAL_LINK_TEXT

In [ ]:
driver = webdriver.Chrome(options=chromeOptions)
driver.get("https://www.screener.in/company/BAJFINANCE/")

wait = WebDriverWait(driver, 10)

# Matches any <a> tag whose visible text CONTAINS the given substring.
# "Balance" matches "Balance Sheet" — useful when the full text is long or may vary.
balance_tab = wait.until(
    EC.element_to_be_clickable((By.PARTIAL_LINK_TEXT, "Balance"))
)
driver.execute_script("arguments[0].scrollIntoView();", balance_tab)
balance_tab.click()

---
## Module 5 — Financial Data Scraper (Capstone)

Scrapes **five NBFC / banking companies** from Screener.in and writes a
`financial_data.csv` containing ten financial metrics per company.

**Companies:** Bajaj Finance · HDFC Bank · Kotak Mahindra Bank · SBI · ICICI Bank  
**Fields:** Market Cap · Current Price · 52W High/Low · Stock P/E · Debt/Equity · ROE · ROCE · Revenue (TTM) · Net Profit (TTM)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import csv

In [ ]:
# Step 1: Configure Chrome — kiosk mode fills the whole screen
chromeOptions = Options()
chromeOptions.add_argument("--kiosk")

In [ ]:
# Step 2: Companies to scrape — ticker symbols used in Screener.in URLs
COMPANIES = {
    "Bajaj Finance":        "BAJFINANCE",
    "HDFC Bank":            "HDFCBANK",
    "Kotak Mahindra Bank":  "KOTAKBANK",
    "State Bank of India":  "SBIN",
    "ICICI Bank":           "ICICIBANK",
}

In [ ]:
# Step 3: Launch Chrome
driver = webdriver.Chrome(options=chromeOptions)

# Step 4: Open CSV and write header row
with open('financial_data.csv', 'w', newline='', encoding='utf-8') as csvfile:

    fieldnames = [
        'Company', 'Market Cap', 'Current Price',
        '52W High', '52W Low', 'Stock P/E',
        'Debt / Equity', 'ROE', 'ROCE',
        'Revenue (TTM)', 'Net Profit (TTM)'
    ]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    # Helper — extract a named metric from the top-ratios section
    def extract_ratio(soup, label):
        for li in soup.select('#top-ratios li'):
            name_span  = li.find('span', class_='name')
            value_span = li.find('span', class_='number')
            if name_span and value_span and label in name_span.get_text():
                return value_span.get_text(strip=True)
        return 'N/A'

    # Step 5: Loop through each company
    for company_name, ticker in COMPANIES.items():
        print(f"\nScraping {company_name} ({ticker}) ...")

        # Step 5a: Navigate to company page and wait for P&L section to load
        driver.get(f"https://www.screener.in/company/{ticker}/")
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "profit-loss"))
        )

        # Step 5b: Scroll incrementally to trigger any lazy-loaded content
        last_height    = driver.execute_script("return window.pageYOffset")
        scroll_increment = 300

        while True:
            driver.execute_script(f"window.scrollTo(0, {scroll_increment});")
            time.sleep(0.8)
            new_height = driver.execute_script("return window.pageYOffset;")
            if new_height == last_height:
                break
            last_height      = new_height
            scroll_increment += 300

        # Step 5c: Parse the page with BeautifulSoup
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        company_data = {'Company': company_name}

        # Step 5d: Extract top-level financial ratios
        company_data['Market Cap']    = extract_ratio(soup, 'Market Cap')
        company_data['Current Price'] = extract_ratio(soup, 'Current Price')
        company_data['Stock P/E']     = extract_ratio(soup, 'Stock P/E')
        company_data['Debt / Equity'] = extract_ratio(soup, 'Debt / Equity')
        company_data['ROE']           = extract_ratio(soup, 'ROE')
        company_data['ROCE']          = extract_ratio(soup, 'ROCE')

        # High / Low is stored as a single combined value — split on '/'
        high_low = extract_ratio(soup, 'High / Low')
        if '/' in high_low:
            parts = [p.strip() for p in high_low.split('/')]
            company_data['52W High'] = parts[0]
            company_data['52W Low']  = parts[1]
        else:
            company_data['52W High'] = high_low
            company_data['52W Low']  = high_low

        # Step 5e: Extract Revenue and Net Profit from the P&L table
        #          Last column holds the most recent TTM figures
        company_data['Revenue (TTM)']    = 'N/A'
        company_data['Net Profit (TTM)'] = 'N/A'

        pl_section = soup.find('section', id='profit-loss')
        if pl_section:
            pl_table = pl_section.find('table')
            if pl_table:
                for row in pl_table.find_all('tr'):
                    cells = row.find_all('td')
                    if not cells:
                        continue
                    row_label = cells[0].get_text(strip=True)
                    if 'Revenue' in row_label or 'Net Sales' in row_label:
                        company_data['Revenue (TTM)'] = cells[-1].get_text(strip=True)
                    elif 'Net Profit' in row_label:
                        company_data['Net Profit (TTM)'] = cells[-1].get_text(strip=True)

        # Step 5f: Write row to CSV and print to console
        writer.writerow(company_data)
        print(company_data)

        # Step 5g: Polite delay before moving to the next company
        time.sleep(3)
        print("Next Company")

driver.quit()
print("\nDone! Data saved to financial_data.csv")